# Demo 3 : PDF-to-Searchable-Index Pipeline

**Module 5A - Hour 3: RAG Fundamentals & Databricks AI Search (Topics 3.1-3.10)**

This notebook demonstrates the full RAG pipeline: document processing, chunking, embeddings, AI Search index creation, and similarity queries - all on the Databricks platform.

Each section has **Notes** (concept + rationale) followed by a **Demo** cell with live code.
At the end: **Learning Conclusion** and **Cleanup** to decommission everything created.

In [0]:
%sql
-- SETUP: Create catalog, schema, and knowledge base table
-- documents table that has CDF and a primary key (both required
-- for Delta Sync vector search indexes).

CREATE CATALOG IF NOT EXISTS module5a_demo3;
CREATE SCHEMA IF NOT EXISTS module5a_demo3.rag;
CREATE VOLUME IF NOT EXISTS module5a_demo3.rag.pdf_documents;

CREATE OR REPLACE TABLE module5a_demo3.rag.knowledge_base (
  doc_id     STRING NOT NULL,
  title      STRING,
  content    STRING,
  category   STRING,
  updated_at TIMESTAMP DEFAULT current_timestamp(),
  CONSTRAINT pk_knowledge_base PRIMARY KEY (doc_id)
) TBLPROPERTIES (
  delta.enableChangeDataFeed = true,
  'delta.feature.allowColumnDefaults' = 'supported'
);



In [0]:
# Setup: Copy the sample PDF to a UC Volume
# ai_parse_document requires binary files in a UC Volume (not workspace files).
# We create the volume, then copy the PDF from the workspace static folder.

spark.sql("CREATE VOLUME IF NOT EXISTS module5a_demo3.rag.pdf_documents")

pdf_source = "file:/Workspace/Users/shriveens@platformatory.com/Data+AI Academy/Module 5A/static/databricks-ebook-a-compact-guide-to-agent-systems.pdf"
pdf_dest = "/Volumes/module5a_demo3/rag/pdf_documents/"

dbutils.fs.cp(pdf_source, pdf_dest, recurse=False)
print("PDF copied to volume")
for f in dbutils.fs.ls(pdf_dest):
    print(f"  {f.name} ({f.size:,} bytes)")

PDF copied to volume
  databricks-ebook-a-compact-guide-to-agent-systems.pdf (2,481,930 bytes)


In [0]:
%sql
-- Setup: Parse the PDF with ai_parse_document and populate the knowledge base
-- ai_parse_document(content, MAP('version', '2.0')) returns a VARIANT with:
--   parsed:document:elements - array of text, tables, figures, titles, etc.
--   parsed:document:pages - page metadata
-- We extract text elements and insert them as documents.

TRUNCATE TABLE module5a_demo3.rag.knowledge_base;

INSERT INTO module5a_demo3.rag.knowledge_base
WITH parsed AS (
  SELECT 
    ai_parse_document(content, MAP('version', '2.0')) AS parsed
  FROM READ_FILES(
    '/Volumes/module5a_demo3/rag/pdf_documents/',
    format => 'binaryFile'
  )
),
elements AS (
  SELECT posexplode(try_cast(parsed:document:elements AS ARRAY<VARIANT>)) AS (pos, element)
  FROM parsed
  WHERE is_variant_null(parsed:error_status)
)
SELECT
  concat('doc-', lpad(cast(row_number() OVER (ORDER BY pos) AS STRING), 3, '0')) AS doc_id,
  CASE 
    WHEN element:type::STRING = 'title' THEN element:content::STRING
    WHEN element:type::STRING = 'section_header' THEN element:content::STRING
    ELSE concat('Section ', cast(pos AS STRING))
  END AS title,
  element:content::STRING AS content,
  element:type::STRING AS category,
  current_timestamp() AS updated_at
FROM elements
WHERE element:type::STRING IN ('text', 'title', 'section_header')
  AND length(coalesce(element:content::STRING, '')) > 30;

SELECT count(*) AS total_docs, collect_set(category) AS doc_types
FROM module5a_demo3.rag.knowledge_base;

total_docs,doc_types
33,"List(section_header, text)"


In [0]:
# Setup: Create a Vector Search endpoint
# This is a managed compute resource that hosts vector indexes.
# We use STANDARD for faster provisioning.
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.vectorsearch import EndpointType
import time

w = WorkspaceClient()
endpoint_name = "demo3_vs_endpoint"

# Create endpoint if it doesn't exist
try:
    ep = w.vector_search_endpoints.get_endpoint(endpoint_name=endpoint_name)
    print(f"Endpoint '{endpoint_name}' already exists: {ep.endpoint_status.state}")
except Exception:
    w.vector_search_endpoints.create_endpoint(
        name=endpoint_name,
        endpoint_type=EndpointType.STANDARD,
    )
    print(f"Creating endpoint '{endpoint_name}'... (takes ~2-3 min)")

# Wait for endpoint to be ready
for i in range(30):
    ep = w.vector_search_endpoints.get_endpoint(endpoint_name=endpoint_name)
    state = str(ep.endpoint_status.state)
    print(f"  State: {state}")
    if "ONLINE" in state:
        print(f"Endpoint '{endpoint_name}' is ready!")
        break
    time.sleep(10)

Creating endpoint 'demo3_vs_endpoint'... (takes ~2-3 min)
  State: EndpointStatusState.ONLINE
Endpoint 'demo3_vs_endpoint' is ready!


In [0]:
# Setup: Create the Vector Search index
# The index is created here (in setup) so that all subsequent demos (3.1, 3.2, 3.3, etc.)
# can use vector_search() to retrieve relevant chunks AUTOMATICALLY instead of hardcoding doc_id.

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.vectorsearch import (
    DeltaSyncVectorIndexSpecRequest, EmbeddingSourceColumn,
    VectorIndexType, PipelineType
)
import time

w = WorkspaceClient()
index_name = "module5a_demo3.rag.knowledge_base_index"
endpoint_name = "demo3_vs_endpoint"

# Create the index (delete and recreate if not ready)
try:
    idx = w.vector_search_indexes.get_index(index_name=index_name)
    if idx.status.ready:
        print(f"Index already exists and is ready!")
    else:
        print("Index not ready. Deleting and recreating...")
        w.vector_search_indexes.delete_index(index_name=index_name)
        time.sleep(15)
        raise Exception("Force recreate")
except Exception:
    print(f"Creating index '{index_name}'...")
    w.vector_search_indexes.create_index(
        name=index_name,
        endpoint_name=endpoint_name,
        primary_key="doc_id",
        index_type=VectorIndexType.DELTA_SYNC,
        delta_sync_index_spec=DeltaSyncVectorIndexSpecRequest(
            source_table="module5a_demo3.rag.knowledge_base",
            embedding_source_columns=[
                EmbeddingSourceColumn(
                    name="content",
                    embedding_model_endpoint_name="databricks-gte-large-en"
                )
            ],
            pipeline_type=PipelineType.TRIGGERED,
            columns_to_sync=["doc_id", "title", "content", "category"],
        ),
    )
    print("Index creation submitted.")

# Wait for index to be ready
for i in range(60):
    idx = w.vector_search_indexes.get_index(index_name=index_name)
    print(f"  Index ready: {idx.status.ready}")
    if idx.status.ready:
        print("Index is ready!")
        break
    time.sleep(10)

# Sync the index (TRIGGERED mode requires manual sync)
print("\nSyncing index...")
try:
    w.vector_search_indexes.sync_index(index_name=index_name)
    time.sleep(5)
except Exception:
    print(f"  Sync already in progress, waiting...")
    time.sleep(30)

print("\nVector Search index is ready for all subsequent demos.")
print("Demos 3.1, 3.2, 3.3, 3.8, and 3.8b will use vector_search() to retrieve automatically.")

Index already exists and is ready!
  Index ready: True
Index is ready!

Syncing index...

Vector Search index is ready for all subsequent demos.
Demos 3.1, 3.2, 3.3, 3.8, and 3.8b will use vector_search() to retrieve automatically.


## 3.1 : Context Engineering vs. Prompt Engineering

### Concepts
* Two ways to adapt a model's behavior:
  * **Prompt engineering**: craft better instructions (change what you ask).
  * **Context engineering**: provide better background knowledge (change what the model sees).
* Context engineering is the primary discipline in practice - most "prompt" issues are actually missing-context issues.
* RAG (Retrieval-Augmented Generation) is the most common context engineering pattern: retrieve relevant documents, inject them as context, then let the model answer.

`ai_query()` lets us demonstrate both approaches. Without context, the model has no knowledge of our specific platform. With context retrieved from our knowledge base, it gives grounded, specific answers.

In [0]:
# 3.1 Demo: Context engineering vs. prompt engineering
# Three approaches, showing why Vector Search is necessary:
#   1. Without context: generic answer
#   2. With hardcoded context: works, but how do you always know the doc_id? You don't.
#   3. With Vector Search: the endpoint decides which chunk is relevant — the right way.

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
index_name = "module5a_demo3.rag.knowledge_base_index"
model = "databricks-meta-llama-3-3-70b-instruct"

def ask(prompt, max_tokens=100):
    """Call ai_query SQL function — same one used throughout this notebook."""
    escaped = prompt.replace("\\", "\\\\").replace("'", "''")
    return spark.sql(f"""
        SELECT ai_query(
            '{model}',
            '{escaped}',
            modelParameters => named_struct('temperature', 0.0, 'max_tokens', {max_tokens})
        )
    """).collect()[0][0]

# --- The user's question — this is the ONLY input ---
user_question = "What is an AI agent?"

# --- 1. Without context (prompt engineering only) ---
print("1. WITHOUT CONTEXT (no retrieval):")
print(ask(user_question + " Answer in 2 sentences."))

# --- 2. With hardcoded context (naive approach) ---
# This works... but how would you always know the doc_id? You wouldn't.
# In a real system, users ask arbitrary questions — you can't pre-map every question to a doc_id.
print("\n2. WITH HARDCODED CONTEXT (naive — you must know the doc_id in advance):")
hardcoded_result = spark.sql("""
    SELECT content FROM module5a_demo3.rag.knowledge_base WHERE doc_id = 'doc-006'
""").collect()

if hardcoded_result:
    context_hardcoded = hardcoded_result[0][0]
    print(f"   Retrieved doc-006 manually: {context_hardcoded[:80]}...")
    print(ask(f'Based on this context: "{context_hardcoded}" {user_question} Answer in 2 sentences.'))
else:
    print("   ❌ doc-006 not found in the table! This proves the point —")
    print("   you can't reliably hardcode a doc_id for every possible question.")

# --- 3. With Vector Search (the right way) ---
# The user's question goes directly to vector_search as query_text.
# The endpoint finds the most relevant chunk automatically — no doc_id needed.
print("\n3. WITH VECTOR SEARCH (the endpoint decides which chunk is relevant):")
results = w.vector_search_indexes.query_index(
    index_name=index_name,
    columns=["doc_id", "title", "content", "category"],
    query_text=user_question,  # ← Same question the user asked
    num_results=1,
)

if results.result and results.result.data_array:
    context = results.result.data_array[0][2]   # content column
    retrieved_doc_id = results.result.data_array[0][0]
    score = results.result.data_array[0][-1]    # similarity score
    print(f"   Vector Search found {retrieved_doc_id} (score: {score:.4f}):")
    print(f"   {context[:80]}...")

    print(f"\n   Answer:")
    print(ask(f'Based on this context: "{context}" {user_question} Answer in 2 sentences.'))
else:
    print("   No results found.")

1. WITHOUT CONTEXT (no retrieval):
An AI agent is a computer program that uses artificial intelligence to perform tasks autonomously, making decisions and taking actions based on its programming, data, and environment. AI agents can be simple or complex, ranging from chatbots and virtual assistants to robots and self-driving cars, and are designed to interact with and adapt to their surroundings to achieve specific goals.

2. WITH HARDCODED CONTEXT (naive — you must know the doc_id in advance):
   Retrieved doc-006 manually: AI agents are intelligent applications designed to automate tasks and enhance hu...
An AI agent is an intelligent application designed to automate tasks and enhance human productivity by analyzing information, making decisions, and taking actions to achieve specific goals. These agents can perform a variety of tasks, such as customer service, campaign generation, and code generation, to free up time and resources for more strategic work, ultimately improving effici

## 3.2 : The RAG Pattern - Indexing Phase vs. Query Phase

### Concepts
RAG has two distinct phases:

* **Indexing phase** (done once or periodically):
  1. Load documents (PDFs, text, structured data)
  2. Chunk documents into smaller pieces
  3. Generate embeddings for each chunk
  4. Store embeddings in a vector search index

* **Query phase** (every user question):
  1. Embed the user's question
  2. Search the vector index for similar chunks (similarity search / ANN)
  3. Inject retrieved chunks as context into the LLM prompt
  4. LLM generates a grounded answer

We demonstrate the query phase using `vector_search()` to retrieve relevant context from our AI Search index, then generate a grounded answer with `ai_query()`.

## 3.3 : Why Retrieval Agents Exist

### Concepts
Three problems that RAG solves:
1. **Knowledge cutoff**: Models are trained on data up to a point. They don't know about events, products, or policies after that date.
2. **Hallucination**: Without grounded context, models fabricate plausible but incorrect answers.
3. **Missing private context**: Models have no access to your organization's internal documents, data, or knowledge base.

RAG addresses all three by retrieving relevant, authoritative context from your own data before generating an answer.

### When to use Feature Store vs Vector Store
Not all data belongs in a vector index:
* **Vector store / RAG**: Best for **unstructured content** (documents, policies, articles) where semantic search is needed.
* **Feature store**: Best for **structured, lookup-based data** (transaction records, pricing, delivery dates) where exact key-based lookup is needed.
* Example: A customer support app that answers shipping questions should use a **feature store table** keyed by `transaction_id` for delivery dates, NOT a vector store of shipping policies. The vector store can't return a specific expected arrival date for a specific order.

### Retrieval Gating
* **Retrieval gating** means requiring a minimum relevance score before using retrieved context.
* If the best chunk's similarity score is below a threshold, the system should **refuse to answer** ("I don't know") instead of guessing.
* This prevents hallucination from weak or irrelevant context and is critical for production RAG systems.
* Combine with a **refusal policy** in the system prompt: "If the retrieved context does not clearly answer the question, say you don't know."

In [0]:
# 3.3 Demo: Hallucination vs. grounded answer
# Without retrieval, the model guesses or fabricates.
# With Vector Search using the USER'S QUESTION, the model stays grounded.
# But what if the question is NOT in the knowledge base? Retrieval gating prevents hallucination.

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
index_name = "module5a_demo3.rag.knowledge_base_index"
model = "databricks-meta-llama-3-3-70b-instruct"

def ask(prompt, max_tokens=50):
    """Call ai_query SQL function — same one used throughout this notebook."""
    escaped = prompt.replace("\\", "\\\\").replace("'", "''")
    return spark.sql(f"""
        SELECT ai_query(
            '{model}',
            '{escaped}',
            modelParameters => named_struct('temperature', 0.0, 'max_tokens', {max_tokens})
        )
    """).collect()[0][0]

# --- The user's question — this is the ONLY input ---

user_question = "What specific evaluation methods and tools does the guide recommend for measuring AI agent system quality?"
RELEVANCE_THRESHOLD = 0.5  # If top score < this, refuse to answer

# --- Without context: the model answers from its training knowledge ---
print("WITHOUT CONTEXT (model uses training knowledge):")
print(ask(user_question + " Answer briefly.", max_tokens=50))

# --- Vector Search retrieves the closest chunk ---
results = w.vector_search_indexes.query_index(
    index_name=index_name,
    columns=["doc_id", "title", "content", "category"],
    query_text=user_question,  # ← Same question the user asked
    num_results=1,
)

if results.result and results.result.data_array:
    context = results.result.data_array[0][2]
    score = results.result.data_array[0][-1]
    retrieved_doc_id = results.result.data_array[0][0]
    print(f"\nVector Search retrieved {retrieved_doc_id} (score: {score:.4f}):")
    print(f"  {context[:120]}...")

    # --- Retrieval gating: check if the score is above threshold ---
    if score < RELEVANCE_THRESHOLD:
        print(f"\n⚠️  Score {score:.4f} < threshold {RELEVANCE_THRESHOLD} — irrelevant!")
        print("REFUSING TO ANSWER (retrieval gating):")
        print('  "I don\'t know. No relevant documents found in the knowledge base."')
        print("\n   Without gating, the model would hallucinate using irrelevant context.")
        print("   With gating, the system correctly refuses — this prevents hallucination.")
    else:
        print(f"\nScore {score:.4f} >= threshold {RELEVANCE_THRESHOLD} — relevant!")
        print(f"\nWITH CONTEXT (grounded):")
        print(ask(f'Context: "{context}"\n\nQuestion: {user_question} Answer briefly.', max_tokens=50))
else:
    print("\nNo relevant documents found.")

WITHOUT CONTEXT (model uses training knowledge):
The guide recommends methods such as precision, recall, F1 score, and mean average precision for evaluating AI agent system quality, as well as tools like metrics libraries and visualization tools.

Vector Search retrieved doc-024 (score: 0.7037):
  Effective evaluation is key to ensuring your AI agent system is delivering high-quality results. Evaluating agent
perfor...

Score 0.7037 >= threshold 0.5 — relevant!

WITH CONTEXT (grounded):
The guide recommends using benchmarks, AI-assisted judges, and human feedback to evaluate AI agent system quality.


## 3.4 : Document Processing Pipeline

### Concepts
The full document processing pipeline:
1. `ai_parse_document(content, MAP('version', '2.0'))` - parse PDFs/images into structured VARIANT (elements: text, tables, figures, titles)
2. `ai_classify(parsed_content, labels, MAP(...))` - categorize documents by type
3. `ai_extract(parsed_content, schema, MAP(...))` - extract structured fields
4. `ai_prep_search(parsed_content)` - chunk for vector search (semantic chunking)

We use a real PDF (Databricks eBook: A Compact Guide to Agent Systems) uploaded to a UC Volume. `ai_parse_document` parses it into structured VARIANT. We show the raw parsed output, then classify and extract key fields.

> `ai_parse_document` requires binary files (PDFs, images, Office docs) stored in a UC Volume. See the SQL reference for the full parsing pipeline.

### Third-Party Python Libraries for Document Processing
While `ai_parse_document` is the Databricks-native approach, the exam may test knowledge of common Python libraries:

| Library | Best For | Why |
|---|---|---|
| `pytesseract` | Scanned images (.jpeg, .png) | OCR wrapper for Tesseract engine; extracts text from images with minimal code |
| `unstructured` | Mixed-format PDFs (text + images) | Handles PDFs with both text and embedded images; least code for mixed content |
| `beautifulsoup` | HTML parsing | Parses HTML/XML documents; not for PDFs or images |
| `scrapy` | Web scraping | Crawls and extracts from web pages; not for local documents |
| `pyquery` | HTML querying | jQuery-like syntax for HTML; not for PDFs or images |
| `flask` | Web framework | Not a document processing library at all |
| `numpy` | Numerical computing | Not a document processing library |

**Exam tip**: If the source documents are **scanned images** (.jpeg, .png), use `pytesseract`. If they are **PDFs with mixed text and images**, use `unstructured`.

In [0]:
%sql
-- 3.4a Demo: Raw ai_parse_document output on the actual PDF
-- This shows what ai_parse_document returns before any processing.
-- The VARIANT contains document.pages (page metadata) and
-- document.elements (text, tables, figures, titles, etc.).

SELECT 
  path,
  size(try_cast(parsed:document:elements AS ARRAY<VARIANT>)) AS num_elements,
  size(try_cast(parsed:document:pages AS ARRAY<VARIANT>)) AS num_pages,
  parsed:metadata AS metadata
FROM (
  SELECT 
    path,
    ai_parse_document(content, MAP('version', '2.0')) AS parsed
  FROM READ_FILES(
    '/Volumes/module5a_demo3/rag/pdf_documents/',
    format => 'binaryFile'
  )
);

-- Show the first 10 elements with their types and a content preview
SELECT 
  element:type::STRING AS element_type,
  substring(element:content::STRING, 1, 120) AS content_preview
FROM (
  SELECT posexplode(try_cast(parsed:document:elements AS ARRAY<VARIANT>)) AS (pos, element)
  FROM (
    SELECT ai_parse_document(content, MAP('version', '2.0')) AS parsed
    FROM READ_FILES(
      '/Volumes/module5a_demo3/rag/pdf_documents/',
      format => 'binaryFile'
    )
  )
  WHERE is_variant_null(parsed:error_status)
)
LIMIT 10;

element_type content_preview text Guide title A Compact Guide
to Al Agents figure figure text databricks page_header A C O M PA C T G U I D E T O A I A G E N T S page_number 2 section_header Contents table Introduction 3 What Is an AI Agent? 5 What Is an AI Age figure

In [0]:
%sql
-- 3.4 Demo: Document processing pipeline - classify and extract
-- Our knowledge_base table was populated from the real PDF using ai_parse_document.
-- Now we classify each document and extract key fields using AI Functions.

SELECT
  doc_id,
  title,
  substring(content, 1, 200) AS content_preview,
  ai_classify(
    content,
    '{"intro":"Introduction or overview of AI agents","architecture":"Agent architecture, reasoning, or design","use_cases":"Business use cases and applications","deployment":"Building, deploying, and evaluating agents","governance":"Governance, security, and compliance"}',
    MAP('version', '2.1', 'enableConfidenceScores', 'true')
  ) AS doc_category,
  ai_extract(
    content,
    '{
      "primary_topic": {"type": "string", "description": "Main subject of the document"},
      "has_code": {"type": "boolean", "description": "Does the document mention code or programming"},
      "key_terms": {"type": "array", "items": {"type": "string"}}
    }',
    MAP('version', '2.1')
  ) AS extracted_fields
FROM module5a_demo3.rag.knowledge_base
ORDER BY doc_id;

doc_id,title,content_preview,doc_category,extracted_fields
doc-001,Scaling GenAI apps into production using AI agent systems,Scaling GenAI apps into production using AI agent systems,"{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":[{""confidence_score"":0.75,""value"":""deployment""}]}","{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":{""has_code"":{""value"":null},""key_terms"":[{""value"":""Scaling GenAI apps""},{""value"":""production""},{""value"":""AI agent systems""}],""primary_topic"":{""value"":""Scaling GenAI apps into production using AI agent systems""}}}"
doc-002,Section 15,"Generative AI holds enormous promise, but for many organizations, transitioning from pilot projects to fully deployed applications remains a significant challenge. According to a recent Economist repo","{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":[{""confidence_score"":0.78,""value"":""deployment""}]}","{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":{""has_code"":{""value"":false},""key_terms"":[{""value"":""Generative AI""},{""value"":""pilot projects""},{""value"":""deployed applications""},{""value"":""enterprises""},{""value"":""GenAI""},{""value"":""infrastructure""},{""value"":""GenAI models""},{""value"":""production-ready""},{""value"":""enterprise data""},{""value"":""AI agent systems""}],""primary_topic"":{""value"":""Challenges in scaling Generative AI applications in enterprises""}}}"
doc-003,"Databricks Mosaic AI: Building high-quality, scalable agent systems","Databricks Mosaic AI: Building high-quality, scalable agent systems","{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":[{""confidence_score"":0.65,""value"":""deployment""}]}","{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":{""has_code"":{""value"":null},""key_terms"":[{""value"":""Databricks""},{""value"":""Mosaic AI""},{""value"":""agent systems""},{""value"":""high-quality""},{""value"":""scalable""}],""primary_topic"":{""value"":""Building high-quality, scalable agent systems using Databricks Mosaic AI""}}}"
doc-004,Section 17,"Databricks Mosaic AI empowers organizations to build and deploy high-quality AI agent systems. Built on lakehouse architecture, Mosaic AI allows secure customization with enterprise data, ensuring acc","{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":[{""confidence_score"":0.68,""value"":""intro""}]}","{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":{""has_code"":{""value"":false},""key_terms"":[{""value"":""high-quality AI agent systems""},{""value"":""lakehouse architecture""},{""value"":""secure customization""},{""value"":""enterprise data""},{""value"":""open source models""},{""value"":""commercial models""},{""value"":""automated tools""},{""value"":""agent systems""},{""value"":""agile development""},{""value"":""robust governance""},{""value"":""data""},{""value"":""models""},{""value"":""visibility""},{""value"":""control""}],""primary_topic"":{""value"":""Databricks Mosaic AI""}}}"
doc-005,Section 23,"In this guide, you’ll gain a comprehensive understanding of AI agents and AI agent systems, their applications and how to build them effectively. Specifically, you’ll learn: ■ The fundamentals of AI a","{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":[{""confidence_score"":0.65,""value"":""intro""}]}","{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":{""has_code"":{""value"":false},""key_terms"":[{""value"":""AI agents""},{""value"":""AI agent systems""},{""value"":""customer service automation""},{""value"":""multi-agent collaboration""},{""value"":""scalable agent systems""},{""value"":""modular agent systems""},{""value"":""enterprise data""},{""value"":""generative AI""},{""value"":""classical AI""},{""value"":""high-performing""},{""value"":""governed""},{""value"":""aligned""}],""primary_topic

## 3.5 : Embeddings & L2 (Euclidean) Distance

### Concepts
* **Embeddings**: Numeric vectors that capture the semantic meaning of text. Similar meanings = similar vectors.
* **L2 (Euclidean) distance**: Measures the straight-line distance between two vectors. Smaller distance = more similar. This is what Databricks Vector Search uses internally.
* **ANN (Approximate Nearest Neighbor)**: The algorithm AI Search uses to find similar vectors efficiently without comparing against all vectors.
* Databricks provides managed embedding models: `databricks-gte-large-en` (1024 dims, 8192 token context) and `databricks-bge-large-en` (1024 dims, 512 token context).

### Embedding Model Selection: Size, Dimensions, and Context Length
When choosing an embedding model, balance **cost/latency** vs **quality**:

| Model | Context Length | Embedding Dimensions | Model Size | Best For |
|---|---|---|---|---|
| Small (e.g., MiniLM) | 512 | 384 | ~0.13 GB | Cost-sensitive, low-latency, short chunks |
| Medium (e.g., BGE Small) | 514 | 768 | ~0.44 GB | Balanced cost/quality |
| Large (e.g., BGE Large) | 2048 | 2560 | ~11 GB | Higher quality, longer context |
| XLarge (e.g., GTE Large) | 32768 | 4096 | ~14 GB | Maximum quality, long documents |

* **Cost/latency priority**: Choose the smallest model whose context length covers your chunk size. If chunks are 512 tokens, a 0.13 GB model with 384 dims is sufficient.
* **Quality priority**: Choose larger models with more dimensions for better semantic discrimination.

### Embedding Model Versioning and Dimension Contracts
* **Vector dimensions are fixed** by the embedding model. You cannot mix vectors of different dimensions in the same index.
* When you change the embedding model (e.g., from 384-dim to 1024-dim), you must **rebuild the entire index** — existing embeddings are incompatible.
* **Never zero-pad** vectors to match a new dimension. This produces garbage vectors that destroy retrieval quality.
* Best practice: **Enforce a contract** — store the model name and dimension alongside the vectors. Reject any mismatched embeddings. Rebuild the index when the model changes.
* Schema changes in source tables can cause silent dimension mismatches if the embedding model is swapped without updating the index.

We use the Databricks SDK to call an embedding model and compute L2 distance manually.

### How Vector Search converts L2 distance to a similarity score (0-1)
* Databricks AI Search uses **L2 (Euclidean) distance** internally.
* Raw L2 distance ranges from 0 (identical) to infinity (completely different).
* Vector Search normalizes embeddings to unit length, then applies: `score = 1 / (1 + L2_normalized²)`
* This maps distance to 0-1: closer vectors → smaller L2 → higher score.
* The first demo cell shows the native Vector Search score; the second cell shows the same calculation by hand.

In [0]:
# 3.5 Demo: Vector Search scores (what the endpoint computes natively)
# Vector Search embeds the query, computes L2 (Euclidean) distance against all docs,
# and returns the top results with their similarity scores (normalized to 0-1).
# The next cell will manually compute the same L2-based score to verify.

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
index_name = "module5a_demo3.rag.knowledge_base_index"

question = "What is an AI agent?"

results = w.vector_search_indexes.query_index(
    index_name=index_name,
    columns=["doc_id", "title", "content"],
    query_text=question,
    num_results=3,
)

print(f"Question: '{question}'")
print(f"\nVector Search results (native — the endpoint does everything):")
if results.result and results.result.data_array:
    for row in results.result.data_array:
        doc_id = row[0]
        title = row[1]
        score = row[-1]
        print(f"  {doc_id} | score: {score:.4f} | {title}")
    print(f"\nTop result score: {results.result.data_array[0][-1]:.4f}")
    print("\nNext cell: manually embed the question + top doc, compute L2 distance,")
    print("and show it matches this Vector Search score.")

Question: 'What is an AI agent?'

Vector Search results (native — the endpoint does everything):
  doc-006 | score: 0.6878 | Section 30
  doc-011 | score: 0.6676 | What Are Use
Cases for an AI
Agent System?
  doc-013 | score: 0.6459 | AI agent system applications across industries

Top result score: 0.6878

Next cell: manually embed the question + top doc, compute L2 distance,
and show it matches this Vector Search score.


In [0]:
# 3.5b Demo: Manual embeddings — the math behind Vector Search
# Vector Search uses L2 (Euclidean) distance, so we compute the same metric by hand.

import math
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# The question
question = "What is an AI agent?"

# The known answer — from doc-006 in our knowledge_base table (parsed from the PDF)
answer = spark.sql("""
    SELECT content FROM module5a_demo3.rag.knowledge_base WHERE doc_id = 'doc-006'
""").collect()[0][0]

print(f"Question: {question}")
print(f"Answer (doc-006): {answer[:100]}...")

# Step 1: Embed both texts using the same model the Vector Search index uses
print("\nEmbedding both texts with databricks-gte-large-en...")
emb_q = w.serving_endpoints.query(name="databricks-gte-large-en", input=[question]).data[0].embedding
emb_a = w.serving_endpoints.query(name="databricks-gte-large-en", input=[answer]).data[0].embedding

print(f"Embedding dimensions: {len(emb_q)}")
print(f"First 5 values (question): {[round(v, 4) for v in emb_q[:5]]}")
print(f"First 5 values (answer):  {[round(v, 4) for v in emb_a[:5]]}")

# Step 2: Compute L2 (Euclidean) distance — what Vector Search uses internally
# L2 = sqrt(sum((a_i - b_i)^2))
l2_raw = math.sqrt(sum((a - b) ** 2 for a, b in zip(emb_q, emb_a)))

print(f"\n--- L2 (EUCLIDEAN) DISTANCE ---")
print(f"L2 distance (raw vectors): {l2_raw:.4f}")

# Step 3: Normalize vectors to unit length, then compute L2
# Vector Search normalizes embeddings before computing L2 distance
norm_q = math.sqrt(sum(a * a for a in emb_q))
norm_a = math.sqrt(sum(b * b for b in emb_a))
q_unit = [a / norm_q for a in emb_q]
a_unit = [b / norm_a for b in emb_a]
l2_norm = math.sqrt(sum((a - b) ** 2 for a, b in zip(q_unit, a_unit)))

print(f"\n--- NORMALIZED L2 (unit vectors) ---")
print(f"||question|| (L2 norm): {norm_q:.4f}")
print(f"||answer|| (L2 norm):   {norm_a:.4f}")
print(f"L2 distance (normalized): {l2_norm:.4f}")
print(f"L2^2 = {l2_norm**2:.4f}")

# Step 4: Convert L2 distance to similarity score (0-1)
# Vector Search formula: score = 1 / (1 + L2_normalized^2)
manual_score = 1 / (1 + l2_norm ** 2)

print(f"\n--- SIMILARITY SCORE ---")
print(f"score = 1 / (1 + L2_normalized^2) = 1 / (1 + {l2_norm**2:.4f}) = {manual_score:.4f}")
print(f"\nPrevious cell Vector Search score for same question vs doc-006: 0.6878")
print(f"Manual L2-based score: {manual_score:.4f}")
match = abs(manual_score - 0.6878) < 0.01
print(f"Match? {'✅ Yes — same calculation, same metric!' if match else '⚠️ Close — floating point precision'}")
print(f"\nVector Search uses L2 (Euclidean) distance internally.")
print(f"The score = 1 / (1 + L2_normalized^2), which maps distance to 0-1 similarity.")
print(f"Closer vectors = smaller L2 = higher score. Farther vectors = larger L2 = lower score.")

Question: What is an AI agent?
Answer (doc-006): AI agents are intelligent applications designed to automate tasks and enhance human productivity. Th...

Embedding both texts with databricks-gte-large-en...
Embedding dimensions: 1024
First 5 values (question): [-0.4893, 0.7437, 0.2803, 0.4641, 0.5952]
First 5 values (answer):  [-0.4199, 1.7646, 0.3364, 0.3428, -0.0075]

--- L2 (EUCLIDEAN) DISTANCE ---
L2 distance (raw vectors): 16.1681

--- NORMALIZED L2 (unit vectors) ---
||question|| (L2 norm): 23.8991
||answer|| (L2 norm):   24.0553
L2 distance (normalized): 0.6743
L2^2 = 0.4547

--- SIMILARITY SCORE ---
score = 1 / (1 + L2_normalized^2) = 1 / (1 + 0.4547) = 0.6874

Previous cell Vector Search score for same question vs doc-006: 0.6878
Manual L2-based score: 0.6874
Match? ✅ Yes — same calculation, same metric!

Vector Search uses L2 (Euclidean) distance internally.
The score = 1 / (1 + L2_normalized^2), which maps distance to 0-1 similarity.
Closer vectors = smaller L2 = higher scor

In [0]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

# Check a known embedding model vs a known LLM
for name in ["databricks-gte-large-en", "databricks-meta-llama-3-3-70b-instruct"]:
    ep = w.serving_endpoints.get(name=name)
    task = ep.task if hasattr(ep, 'task') else "N/A"
    print(f"\n{name}")
    print(f"  task: {task}")
    if ep.config and ep.config.served_entities:
        for e in ep.config.served_entities:
            if hasattr(e, 'foundation_model') and e.foundation_model:
                fm = e.foundation_model
                print(f"  foundation_model.name: {fm.name}")
                print(f"  foundation_model.display_name: {fm.display_name}")
                print(f"  foundation_model.description: {fm.description}")


databricks-gte-large-en
  task: llm/v1/embeddings
  foundation_model.name: system.ai.gte_large_en_v1_5
  foundation_model.display_name: GTE Large (En)
  foundation_model.description: General text embeddings (GTE) can map any text to a low-dimensional dense vector which can be used for tasks like retrieval, classification, clustering, or semantic search. And it also can be used in vector databases for LLMs.

databricks-meta-llama-3-3-70b-instruct
  task: llm/v1/chat
  foundation_model.name: system.ai.llama_v3_3_70b_instruct
  foundation_model.display_name: Meta Llama 3.3 70B Instruct
  foundation_model.description: Llama 3.3 is a state-of-the-art 70B parameter dense language model trained and released by Meta. The model supports a context length of 128K tokens. The model is optimized for multilingual dialogue use cases and aligned with human preferences for helpfulness and safety. It is not intended for use in languages other than English. Meta Llama 3.3 is licensed under the Meta Llam

## 3.6 : Chunking — What It Is & Strategy Considerations

### What is chunking?
* **Chunking** = splitting long documents into smaller pieces (chunks) before embedding.
* **Why?** Embedding models have context limits (e.g., 512-8192 tokens). A 50-page PDF won't fit in one embedding. Smaller chunks also give more precise retrieval — you find the exact paragraph, not a whole chapter.
* **When to use it?** When source documents are long (articles, manuals, full PDFs). Short documents (a few sentences) don't need chunking.

### Why didn't our main pipeline (3.1-3.3) use chunking?
* Our `ai_parse_document` already splits the PDF into individual text elements (paragraphs, section headers, titles). Each row in `knowledge_base` is already a small "chunk" — typically 1-3 sentences.
* So chunking was implicitly done by `ai_parse_document` itself. We indexed those elements directly into Vector Search.
* **Chunking is an extra layer** you can add after `ai_parse_document` and before index creation — useful when the parsed elements are still too long or when you want overlap between chunks for better retrieval continuity.

### Chunking strategy considerations
* **Chunk size**: Balance between too small (loses context) and too large (dilutes relevance, hits embedding model token limits).
* **Overlap**: 10-20% overlap between chunks prevents losing context at boundaries.
* **Granularity trade-off**: Smaller chunks = more precise retrieval but less context per chunk. Larger chunks = more context but less precise.
* **"Lost in the middle"**: Models pay less attention to content in the middle of long contexts. Keep chunks focused.

We demonstrate manual chunking by splitting documents into sentences using SQL string functions (3.6 Demo). In production, use `ai_prep_search` (demonstrated in 3.7b) for semantic chunking that respects paragraph and section boundaries — it also generates `chunk_to_embed` and `chunk_to_retrieve` automatically.

### Retrieval Budget and Context Compression
* **Retrieval budget**: Limit the number and size of chunks fed to the LLM based on dynamic relevance scores.
* If you retrieve 10 chunks but only have token budget for 5, keep the top-5 by relevance score.
* **Context compression**: Summarize or compress retrieved context before passing it to the generation LLM.
* This prevents **token limit errors** when the combined context + prompt exceeds the model's context window.
* Common symptom: the model truncates answers because the input context is too long.
* Mitigations: (a) reduce chunk size, (b) reduce number of retrieved records (lower k), (c) compress/summarize context before generation.

In [0]:
%sql
-- 3.6 Demo: Manual chunking of documents
--
-- HOW IT WORKS:
--   split(content, '. ') splits text at every '. ' (period + space).
--   Each sentence becomes its own row = one chunk.
--
-- WHY NO CHUNK SIZE PARAMETER?
--   We are NOT splitting by a fixed token or character count.
--   We are splitting by SENTENCE BOUNDARIES — each sentence IS a chunk.
--   The chunk size is whatever the sentence length happens to be.
--   Some chunks are short (57 chars), some are long (900+ chars).
--   The chunk_length column shows this variation.
--
-- LIMITATIONS of this manual approach:
--   1. Naive splitter: '. ' breaks on abbreviations like 'e.g. ' or 'Inc. '
--   2. No overlap between chunks (can lose context at boundaries)
--   3. No awareness of paragraphs or section headers
--   4. No enrichment (title, category not added to the chunk text)
--
-- The next cell (3.6b) shows ai_prep_search which handles all of this
-- automatically — semantic chunking + chunk_to_embed + chunk_to_retrieve.

WITH sentences AS (
  SELECT
    doc_id,
    title,
    sentence,
    row_number() OVER (PARTITION BY doc_id ORDER BY 1) AS sentence_num
  FROM module5a_demo3.rag.knowledge_base
  LATERAL VIEW explode(split(content, '\\. ')) AS sentence
  WHERE sentence != ''
)
SELECT
  doc_id,
  title,
  sentence_num AS chunk_id,
  length(sentence) AS chunk_length,
  sentence AS chunk_text
FROM sentences
ORDER BY doc_id, sentence_num
LIMIT 20;

doc_id,title,chunk_id,chunk_length,chunk_text
doc-001,Scaling GenAI apps into production using AI agent systems,1,57,Scaling GenAI apps into production using AI agent systems
doc-002,Section 15,1,162,"Generative AI holds enormous promise, but for many organizations, transitioning from pilot projects to fully deployed applications remains a significant challenge"
doc-002,Section 15,2,239,"According to a recent Economist report, “Unlocking Enterprise AI,” 85% of global enterprises are already using GenAI, a number expected to reach 99% by 2027. However, many organizations face challenges in scaling these projects effectively"
doc-002,Section 15,3,170,"The report also states that only 22% of enterprises are confident their infrastructure is ready for AI, and just 37% believe their GenAI models are truly production-ready"
doc-002,Section 15,4,279,"These gaps highlight the need for a robust infrastructure and advanced tools to ensure GenAI applications meet the high standards required for success. Many GenAI projects fall short by failing to integrate with enterprise data, which can lead to inaccurate or irrelevant results"
doc-002,Section 15,5,188,"To fully unlock the potential of GenAI, businesses need more than stand-alone models — they need comprehensive AI agent systems that are tailored to their specific data and business needs."
doc-003,"Databricks Mosaic AI: Building high-quality, scalable agent systems",1,67,"Databricks Mosaic AI: Building high-quality, scalable agent systems"
doc-004,Section 17,1,93,Databricks Mosaic AI empowers organizations to build and deploy high-quality AI agent systems
doc-004,Section 17,2,135,"Built on lakehouse architecture, Mosaic AI allows secure customization with enterprise data, ensuring accurate, domain-specific outputs"
doc-004,Section 17,3,349,"It offers a secure way to connect to open source or commercial models, providing the flexibility to choose the best-fit solutions. With automated tools to evaluate and improve agent systems quickly, Mosaic AI ensures agile development and robust governance across every component, from data to models, giving businesses full visibility and control."


## 3.7 : chunk_to_embed vs. chunk_to_retrieve

### Two types of chunk text
Once you have chunks (from 3.6), you can use **different text** for embedding vs. retrieval:

| Aspect | chunk_to_embed | chunk_to_retrieve |
|---|---|---|
| Purpose | Generate the embedding vector | Return to the LLM as context |
| Content | Enriched (title + category + text) | Clean raw text only |
| Why? | Extra context helps findability | Clean text avoids confusing the LLM |
| Example | `"AI Agents \| text \| AI agents are intelligent applications..."` | `"AI agents are intelligent applications..."` |

* **chunk_to_embed**: Enriched with title, category, or keywords to improve semantic matching during search.
* **chunk_to_retrieve**: Clean raw content returned to the LLM — no metadata noise.
* Separating them improves both retrieval accuracy (better embeddings) and answer quality (cleaner context).

This demo builds on the manual chunking from 3.6, showing the distinction between embed text and retrieve text.

In [0]:
%sql
-- 3.7 Demo: chunk_to_embed vs. chunk_to_retrieve
-- We create a chunked version of our knowledge base with two text columns:
-- - chunk_to_embed: enriched with title + category for better findability
-- - chunk_to_retrieve: clean raw content for the LLM

CREATE OR REPLACE TABLE module5a_demo3.rag.document_chunks AS
WITH sentences AS (
  SELECT
    doc_id,
    title,
    category,
    sentence,
    row_number() OVER (PARTITION BY doc_id ORDER BY 1) AS sentence_num
  FROM module5a_demo3.rag.knowledge_base
  LATERAL VIEW explode(split(content, '\\. ')) AS sentence
  WHERE sentence != ''
)
SELECT
  concat(doc_id, '-', lpad(cast(sentence_num AS STRING), 3, '0')) AS chunk_id,
  doc_id,
  title,
  category,
  -- chunk_to_embed: enriched with title + category for findability
  concat(title, ' | ', category, ' | ', sentence) AS chunk_to_embed,
  -- chunk_to_retrieve: clean raw text for the LLM
  sentence AS chunk_to_retrieve
FROM sentences;

SELECT chunk_id, chunk_to_embed, chunk_to_retrieve
FROM module5a_demo3.rag.document_chunks
ORDER BY chunk_id
LIMIT 10;

chunk_id,chunk_to_embed,chunk_to_retrieve
doc-001-001,Scaling GenAI apps into production using AI agent systems | section_header | Scaling GenAI apps into production using AI agent systems,Scaling GenAI apps into production using AI agent systems
doc-002-001,"Section 15 | text | Generative AI holds enormous promise, but for many organizations, transitioning from pilot projects to fully deployed applications remains a significant challenge","Generative AI holds enormous promise, but for many organizations, transitioning from pilot projects to fully deployed applications remains a significant challenge"
doc-002-002,"Section 15 | text | According to a recent Economist report, “Unlocking Enterprise AI,” 85% of global enterprises are already using GenAI, a number expected to reach 99% by 2027. However, many organizations face challenges in scaling these projects effectively","According to a recent Economist report, “Unlocking Enterprise AI,” 85% of global enterprises are already using GenAI, a number expected to reach 99% by 2027. However, many organizations face challenges in scaling these projects effectively"
doc-002-003,"Section 15 | text | The report also states that only 22% of enterprises are confident their infrastructure is ready for AI, and just 37% believe their GenAI models are truly production-ready","The report also states that only 22% of enterprises are confident their infrastructure is ready for AI, and just 37% believe their GenAI models are truly production-ready"
doc-002-004,"Section 15 | text | These gaps highlight the need for a robust infrastructure and advanced tools to ensure GenAI applications meet the high standards required for success. Many GenAI projects fall short by failing to integrate with enterprise data, which can lead to inaccurate or irrelevant results","These gaps highlight the need for a robust infrastructure and advanced tools to ensure GenAI applications meet the high standards required for success. Many GenAI projects fall short by failing to integrate with enterprise data, which can lead to inaccurate or irrelevant results"
doc-002-005,"Section 15 | text | To fully unlock the potential of GenAI, businesses need more than stand-alone models — they need comprehensive AI agent systems that are tailored to their specific data and business needs.","To fully unlock the potential of GenAI, businesses need more than stand-alone models — they need comprehensive AI agent systems that are tailored to their specific data and business needs."
doc-003-001,"Databricks Mosaic AI: Building high-quality, scalable agent systems | section_header | Databricks Mosaic AI: Building high-quality, scalable agent systems","Databricks Mosaic AI: Building high-quality, scalable agent systems"
doc-004-001,Section 17 | text | Databricks Mosaic AI empowers organizations to build and deploy high-quality AI agent systems,Databricks Mosaic AI empowers organizations to build and deploy high-quality AI agent systems
doc-004-002,"Section 17 | text | Built on lakehouse architecture, Mosaic AI allows secure customization with enterprise data, ensuring accurate, domain-specific outputs","Built on lakehouse architecture, Mosaic AI allows secure customization with enterprise data, ensuring accurate, domain-specific outputs"
doc-004-003,"Section 17 | text | It offers a secure way to connect to open source or commercial models, providing the flexibility to choose the best-fit solutions. With automated tools to evaluate and improve agent systems quickly, Mosaic AI ensures agile development and robust governance across every component, from data to models, giving businesses full visibility and control.","It offers a secure way to connect to open source or commercial models, providing the flexibility to choose the best-fit solutions. With automated tools to evaluate and improve agent systems quickly, Mosaic AI ensures agile development and robust governance across every component, from data to models, giving businesses full visibility 

In [0]:
%sql
-- 3.7b Demo: ai_prep_search — production-grade automatic chunking + enrichment
--
-- The 3.7 Demo above showed MANUAL chunk_to_embed vs chunk_to_retrieve (title | category | sentence).
-- ai_prep_search() does ALL of that in ONE function call:
--   1. Splits content into semantic chunks (respects sentence/paragraph/section boundaries)
--   2. Generates chunk_to_embed: enriched with metadata (title, subject, company, etc.)
--   3. Generates chunk_to_retrieve: clean raw text for the LLM
--   4. Assigns chunk_id and chunk_position automatically
--
-- Compare the chunk_to_embed column (enriched) vs chunk_to_retrieve column (clean):
--   - chunk_to_embed has metadata PREFIX added by ai_prep_search (e.g. "Company: Databricks")
--   - chunk_to_retrieve is the raw content without enrichment
--   - embed_len > retrieve_len because of the added context
--
-- PRODUCTION PIPELINE CONNECTION:
--   In production, the output of ai_prep_search is written to a Delta table
--   and THAT table becomes the source_table_name for the Vector Search index.
--   Our demo index uses knowledge_base directly (ai_parse_document already produced
--   paragraph-level chunks), but ai_prep_search adds enrichment that improves retrieval.
--
-- NOTE: Our knowledge_base rows are already paragraph-level (from ai_parse_document),
-- so ai_prep_search keeps each as 1 chunk. For longer source documents (full PDFs),
-- it would split into multiple chunks with chunk_position 0, 1, 2, ...

WITH prepped AS (
  SELECT 
    doc_id,
    title,
    ai_prep_search(content) AS result
  FROM module5a_demo3.rag.knowledge_base
  WHERE length(content) > 200
  LIMIT 5
)
SELECT
  prepped.doc_id,
  prepped.title,
  chunk.value:chunk_id::STRING AS chunk_id,
  chunk.value:chunk_position::INT AS chunk_position,
  length(chunk.value:chunk_to_retrieve::STRING) AS retrieve_len,
  length(chunk.value:chunk_to_embed::STRING) AS embed_len,
  substring(chunk.value:chunk_to_retrieve::STRING, 1, 80) AS chunk_to_retrieve_preview,
  substring(chunk.value:chunk_to_embed::STRING, 1, 120) AS chunk_to_embed_preview
FROM
  prepped,
  LATERAL variant_explode(prepped.result:document.contents) AS chunk
ORDER BY prepped.doc_id, chunk_position;

doc_id,title,chunk_id,chunk_position,retrieve_len,embed_len,chunk_to_retrieve_preview,chunk_to_embed_preview
doc-002,Section 15,8640c2ac0b84424da08fd7c2f76850df_0,0,1048,1222,"Generative AI holds enormous promise, but for many organizations, transitioning",Report Title: Unlocking Enterprise AI Source: The Economist Subject: Generative AI Analyst summary of Economist report
doc-004,Section 17,b4b99da2dd4f4b659f5c2d641c5163f7_0,0,583,756,Databricks Mosaic AI empowers organizations to build and deploy high-quality AI,Company: Databricks Product: Mosaic AI Subject: Mosaic AI Technology: lakehouse architecture Databricks product overvie
doc-005,Section 23,8d82d017402d4e7ab4d6a20d60ca4543_0,0,935,1057,"In this guide, you’ll gain a comprehensive understanding of AI agents and AI ag",Document Type: Guide Subject: AI agents and AI agent systems A corporate AI agents guide from the early 2020s. Content
doc-006,Section 30,5e149f626a6a48dba24f7890b6307bfb_0,0,1021,1152,AI agents are intelligent applications designed to automate tasks and enhance h,"Subject: AI agents Created by an unknown source, this informational overview covers contemporary AI agent applications."
doc-008,Section 43,ea8865bd435d4d94aeef6cb50471de64_0,0,294,438,I should look up the recent order and our return policies. 1.Look up the user',"Action: Start return process and send shipping label to user. Internal support note on handling a customer's return, cu"


## 3.8 : Creating & Syncing an AI Search Index

### Concepts
* **Endpoint**: Compute resource that hosts indexes (Standard or Storage-Optimized).
* **Delta Sync Index**: Automatically syncs with a source Delta table. Databricks computes embeddings for you.
* **Key parameters**:
  * `endpoint_name`: Vector search endpoint to host the index
  * `source_table_name`: Delta table with CDF and primary key
  * `embedding_model_endpoint_name`: Model for computing embeddings (e.g., `databricks-gte-large-en`)
  * `pipeline_type`: `TRIGGERED` (manual sync) or `CONTINUOUS` (auto-sync)
* **Change Data Feed (CDF)**: Required for Delta Sync - enables incremental updates.
* **Primary Key**: Required constraint on the source table.

We created both the endpoint and the index in setup, so all demos (3.1 onward) can use `vector_search()` to retrieve relevant chunks. This demo explores different query parameters and similarity scores.

### Source Table: Raw vs. ai_prep_search Output
* Our demo index uses `module5a_demo3.rag.knowledge_base` directly — this works because `ai_parse_document` already produced paragraph-level chunks.
* In production, you should use the output of `ai_prep_search` (see 3.7b) as the source table — it adds enrichment (`chunk_to_embed` with metadata) and produces `chunk_to_retrieve` (clean text) automatically.
* The enriched `chunk_to_embed` column improves retrieval recall; the clean `chunk_to_retrieve` column ensures the LLM gets faithful context.

### Standard vs Storage-Optimized Endpoints
* **Standard**: Faster provisioning, good for smaller indexes (< ~25M vectors). Higher query throughput.
* **Storage-Optimized**: Designed for **large-scale indexes** (100M+ vectors). Lower storage cost per vector, slightly higher latency but handles massive inventories.
* Rule of thumb: 100M+ items or latency-sensitive at scale -> Storage-Optimized.

### Hybrid Search and Reranking
* **Hybrid search** combines **vector similarity** (semantic) with **lexical search** (BM25 / keyword matching) to improve retrieval quality.
* **BM25 (Best Matching 25)**: A ranking function that scores documents by keyword frequency and inverse document frequency. It excels at exact-term matching (product names, SKUs, IDs).
* **Why combine?**: Vector search catches semantic similarity ("AI agent" matches "intelligent assistant"). BM25 catches exact terms ("ORD-001" matches "ORD-001"). Together they cover both.
* **Reranking**: A second-pass model that re-scores the top-k results from hybrid search to produce a more accurate final ranking.
* **When to use**: Turn on hybrid search + reranking when accuracy matters and you can afford slightly higher latency. Turn off for maximum speed.
* For 100M items where latency is critical: use **Storage-Optimized** + GTE Large + hybrid search + reranking — the storage-optimized endpoint keeps latency acceptable at scale.

### Vector Search Data Format
* Vector Search requires **one row per chunk** with a **unique primary key**.
* If your dataframe has an array of chunks per document (one row, multiple chunks), you must **flatten** it: one chunk per row, each with its own unique ID.
* Do NOT store chunks as individual JSON files in UC volumes — Delta tables with one row per chunk are the performant format for indexing.

In [0]:
# 3.8 Demo: Creating, Syncing & Querying the AI Search index
# Recap what was created, show syncing, then explore query parameters.

from databricks.sdk import WorkspaceClient
import time

w = WorkspaceClient()
index_name = "module5a_demo3.rag.knowledge_base_index"

# === PART 1: What was created in setup ===
print("=== INDEX CONFIGURATION (created in setup) ===")
idx = w.vector_search_indexes.get_index(index_name=index_name)
print(f"  Index name: {index_name}")
print(f"  Indexed rows: {idx.status.indexed_row_count}")
print(f"  Ready: {idx.status.ready}")
print(f"  Key parameters used in setup:")
print(f"    - endpoint_name: demo3_vs_endpoint")
print(f"    - source_table: module5a_demo3.rag.knowledge_base")
print(f"    - embedding_model: databricks-gte-large-en")
print(f"    - pipeline_type: TRIGGERED (manual sync)")
print(f"    - primary_key: doc_id")
print(f"    - CDF enabled: yes (delta.enableChangeDataFeed = true)")
print(f"    - Production note: use ai_prep_search output (3.7b) as source for enrichment")

# === PART 2: Sync the index (TRIGGERED mode requires manual sync) ===
print("\n=== SYNCING INDEX (TRIGGERED mode = manual) ===")
print("Triggering sync...")
try:
    w.vector_search_indexes.sync_index(index_name=index_name)
    print("Sync triggered. This updates embeddings for any changed rows in the source table.")
    print("In CONTINUOUS mode, this happens automatically without manual sync.")
except Exception as e:
    print(f"  Sync already in progress or failed: {str(e)[:80]}")
print("\n  Why syncing matters:")
print("    - Source table changed? Re-sync to update embeddings.")
print("    - New documents added? Re-sync to index them.")
print("    - Without sync, the index has STALE embeddings (wrong retrieval scores).")

time.sleep(3)

# === PART 3: Query with different parameters ===
print("\n=== QUERY 1: 'What is an AI agent?' (num_results=3) ===")
results = w.vector_search_indexes.query_index(
    index_name=index_name,
    columns=["doc_id", "title", "content", "category"],
    query_text="What is an AI agent?",
    num_results=3,
)
for row in (results.result.data_array if results.result and results.result.data_array else []):
    score = row[-1]
    print(f"  Score: {score:.4f} | {row[1]} | {row[2][:80]}...")

print("\n=== QUERY 2: 'How do you build an agent system?' (num_results=3) ===")
results = w.vector_search_indexes.query_index(
    index_name=index_name,
    columns=["doc_id", "title", "content", "category"],
    query_text="How do you build an agent system?",
    num_results=3,
)
for row in (results.result.data_array if results.result and results.result.data_array else []):
    score = row[-1]
    print(f"  Score: {score:.4f} | {row[1]} | {row[2][:80]}...")

print("\n=== QUERY 3: 'What are the use cases for AI agents?' (num_results=5) ===")
results = w.vector_search_indexes.query_index(
    index_name=index_name,
    columns=["doc_id", "title", "content", "category"],
    query_text="What are the use cases for AI agents?",
    num_results=5,
)
for row in (results.result.data_array if results.result and results.result.data_array else []):
    score = row[-1]
    print(f"  Score: {score:.4f} | {row[1]} | {row[2][:80]}...")

print("\n=== OBSERVATIONS ===")
print("1. Creation: endpoint + index created in setup with CDF + primary key")
print("2. Sync: TRIGGERED mode requires manual sync; CONTINUOUS syncs automatically")
print("3. Different questions retrieve different chunks — the endpoint decides, not us")
print("4. Scores range from 0-1; higher = more semantically similar")
print("5. Notice score drop-off: top results are much more relevant than lower ones")

=== INDEX CONFIGURATION (created in setup) ===
  Index name: module5a_demo3.rag.knowledge_base_index
  Indexed rows: 33
  Ready: True
  Key parameters used in setup:
    - endpoint_name: demo3_vs_endpoint
    - source_table: module5a_demo3.rag.knowledge_base
    - embedding_model: databricks-gte-large-en
    - pipeline_type: TRIGGERED (manual sync)
    - primary_key: doc_id
    - CDF enabled: yes (delta.enableChangeDataFeed = true)
    - Production note: use ai_prep_search output (3.7b) as source for enrichment

=== SYNCING INDEX (TRIGGERED mode = manual) ===
Triggering sync...
Sync triggered. This updates embeddings for any changed rows in the source table.
In CONTINUOUS mode, this happens automatically without manual sync.

  Why syncing matters:
    - Source table changed? Re-sync to update embeddings.
    - New documents added? Re-sync to index them.
    - Without sync, the index has STALE embeddings (wrong retrieval scores).

=== QUERY 1: 'What is an AI agent?' (num_results=3) ===

## 3.9 : Managed RAG (Knowledge Assistant) vs. Custom RAG (Agent Framework + AI Search)

### Concepts

| Dimension | Managed RAG (Knowledge Assistant) | Custom RAG (Agent Framework + AI Search) |
|---|---|---|
| Setup time | Minutes (no-code UI) | Hours to days (code) |
| Customizability | Limited (pre-built) | Full control |
| Chunking | Automatic | You control strategy |
| Retrieval | Automatic | You tune search parameters |
| Model choice | Limited to configured models | Any model via endpoints |
| When to use | Quick POC, simple Q&A | Production, complex requirements |

* **Knowledge Assistant** (Agent Bricks): point at a UC volume of documents, get a chatbot. Best for fast time-to-value.
* **Custom RAG**: build your own pipeline with Agent Framework + AI Search. Best when you need custom chunking, filtering, or multi-step retrieval.

This demo is a **Custom RAG** pipeline: we controlled the table schema, chunking, embedding model, and index parameters.

## 3.10 : Unity Catalog Governance Across the Retrieval Pipeline

### Concepts
Every component of the RAG pipeline is governed by Unity Catalog:
* **Volumes**: Source documents stored in UC Volumes (governed storage)
* **Tables**: Parsed/chunked documents are Delta tables (table-level ACLs)
* **Indexes**: AI Search indexes are UC entities (create/query permissions)
* **Endpoints**: Vector Search endpoints have separate ACLs (CAN_USE)
* **Models**: Embedding and LLM endpoints have their own access controls
* **Lineage**: UC tracks data flow from source documents through chunks to index to answer

All permissions are managed in one place - no separate governance system for AI components.

## Learning Conclusion

### What we demonstrated

| Topic | What was demoed | Key Takeaway |
|---|---|---|
| 3.1 | `ai_query` with and without context (vector_search retrieval) | Context engineering > prompt engineering for grounded answers; Vector Search retrieves the right chunk automatically |
| 3.2 | RAG query phase (vector_search + generate) | RAG = indexing phase + query phase; vector_search() retrieves relevant context |
| 3.3 | Hallucinated vs. grounded answer (vector_search retrieval) | RAG prevents hallucination; feature store for structured data, vector store for unstructured; retrieval gating with relevance threshold |
| 3.4 | `ai_classify` + `ai_extract` on documents | Document pipeline: parse, classify, extract; pytesseract for images, `unstructured` for mixed PDFs |
| 3.5 | Embedding model + L2 (Euclidean) distance (manual) vs Vector Search score | Embeddings capture meaning; Vector Search uses L2 distance with `score = 1/(1+L2²)`; manual calculation matches Vector Search score |
| 3.6 | What is chunking + manual sentence-level chunking | Chunking splits long docs for precise retrieval; `ai_parse_document` already chunks implicitly; chunk size/overlap affect quality |
| 3.7 | `chunk_to_embed` vs `chunk_to_retrieve` | Enriched text for findability, clean text for the LLM |
| 3.7b | `ai_prep_search` automatic chunking + enrichment | One function does semantic chunking + generates chunk_to_embed (enriched) and chunk_to_retrieve (clean); output is the production source for Vector Search index |
| 3.8 | Vector Search endpoint + Delta Sync index + query | Standard vs Storage-Optimized; hybrid search + BM25 + reranking; one chunk per row |
| 3.8b | Full RAG pipeline: vector search retrieval + LLM generation | Vector Search automatically finds relevant chunks (no hardcoded doc_id); RAG answers are grounded vs. generic no-context answers |
| 3.9 | Managed vs. Custom RAG comparison | Knowledge Assistant for speed, Custom RAG for control |
| 3.10 | UC governance overview | Every RAG component governed by Unity Catalog |

### Key principles
* **RAG = retrieve + generate**: Get relevant context from your data, then let the LLM answer.
* **Chunking matters**: How you split documents affects retrieval quality.
* **AI Search is managed**: Databricks computes embeddings, maintains the index, and serves queries.
* **Everything is governed**: Unity Catalog controls access to every component.
* **Start managed, go custom when needed**: Knowledge Assistant first, Custom RAG when you need control.

In [0]:
# CLEANUP: Decommission everything created in this demo
from databricks.sdk import WorkspaceClient
import time

w = WorkspaceClient()

index_name = "module5a_demo3.rag.knowledge_base_index"
endpoint_name = "demo3_vs_endpoint"

# 1. Delete the vector search index
print("Deleting index...")
try:
    w.vector_search_indexes.delete_index(index_name=index_name)
    print(f"  Deleted index: {index_name}")
except Exception as e:
    print(f"  Index deletion: {e}")

time.sleep(3)

# 2. Delete the vector search endpoint
print("Deleting endpoint...")
try:
    w.vector_search_endpoints.delete_endpoint(endpoint_name=endpoint_name)
    print(f"  Deleted endpoint: {endpoint_name}")
except Exception as e:
    print(f"  Endpoint deletion: {e}")

# 3. Drop the schema and catalog (cascades to tables)
print("Dropping schema and catalog...")
spark.sql("DROP VOLUME IF EXISTS module5a_demo3.rag.pdf_documents")
spark.sql("DROP SCHEMA IF EXISTS module5a_demo3.rag CASCADE")
spark.sql("DROP CATALOG IF EXISTS module5a_demo3 CASCADE")
print("  Dropped schema and catalog")

print("\nCleanup complete!")

Deleting index...
  Deleted index: module5a_demo3.rag.knowledge_base_index
Deleting endpoint...
  Deleted endpoint: demo3_vs_endpoint
Dropping schema and catalog...
  Dropped schema and catalog

Cleanup complete!
